# sklearn Pipelines — End-to-End Preprocessing

All the techniques from previous notebooks — imputation, encoding, scaling, transformation — only work correctly in production if they are applied *consistently* and *without leakage*. The sklearn `Pipeline` and `ColumnTransformer` API enforce both automatically.

### What you will learn

| Section | Topic |
|---------|-------|
| 1 | Why pipelines exist — the manual approach breaks |
| 2 | `Pipeline` — chaining sequential steps |
| 3 | `ColumnTransformer` — applying different transforms per column type |
| 4 | Full preprocessing + model pipeline |
| 5 | Cross-validation with Pipeline (zero leakage by construction) |
| 6 | Inspecting, saving, and loading a Pipeline |
| 7 | Custom transformers |

**Dataset:** Titanic + California Housing — one classification, one regression.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer
)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import classification_report, r2_score
from sklearn.datasets import fetch_california_housing
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.dpi'] = 110

print('Setup complete.')

Setup complete.


---
## Section 1 — Why Pipelines Exist

The manual approach to preprocessing has two failure modes:

### Failure 1: Data leakage
```python
# WRONG — scaler sees test data during fit
scaler.fit(X)                    # fits on full dataset
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
```

### Failure 2: Training/serving skew
```python
# At training time:
imputer.fit_transform(X_train)

# At serving time (a new single row arrives):
# You have to remember every transform, in order, with the right parameters
# Forgetting one → silent incorrect predictions in production
```

A `Pipeline` solves both: it bundles all steps into a single object that can be fit, evaluated, saved, and deployed as one unit.

In [2]:
# Load Titanic
df = sns.load_dataset('titanic')

# Select a clean feature set
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class']
target   = 'survived'

df_model = df[features + [target]].copy()
print(f'Shape: {df_model.shape}')
print('\nMissing values:')
print(df_model.isnull().sum()[df_model.isnull().sum() > 0])

URLError: <urlopen error [WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。>

In [ ]:
X = df_model.drop(columns=target)
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Survival rate — train: {y_train.mean():.2%}  test: {y_test.mean():.2%}')

---
## Section 2 — Pipeline: Chaining Sequential Steps

A `Pipeline` applies a sequence of transformers followed by a final estimator. Each step must implement `fit` and `transform`; the final step must implement `fit` (and `predict` for supervised tasks).

```python
pipe = Pipeline([
    ('step_name_1', Transformer1()),
    ('step_name_2', Transformer2()),
    ('model',       Estimator()),
])
```

Calling `pipe.fit(X_train, y_train)` calls each transformer's `fit_transform` in sequence, then `model.fit` on the final output.

In [ ]:
# Simple pipeline for a single numeric feature to demonstrate the concept
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])

age_train = X_train[['age', 'fare']]
age_transformed = numeric_pipe.fit_transform(age_train)

print('Input:')
print(pd.DataFrame(age_train).describe().round(2))
print('\nAfter Pipeline (impute → scale):')
print(pd.DataFrame(age_transformed, columns=['age', 'fare']).describe().round(4))

---
## Section 3 — ColumnTransformer: Different Steps per Column Type

Real datasets have mixed types: numeric columns need imputation + scaling; categorical columns need imputation + encoding. `ColumnTransformer` applies different transformers to different column subsets and concatenates the results.

```python
ct = ColumnTransformer([
    ('name_a', transformer_a, ['col1', 'col2']),  # explicit column list
    ('name_b', transformer_b, ['col3']),
], remainder='drop')  # or 'passthrough' to keep un-specified columns
```

In [ ]:
# Identify column types
numeric_features     = ['age', 'fare', 'sibsp', 'parch']
low_card_cat_features = ['sex', 'embarked']   # OHE — low cardinality
ordinal_features      = ['pclass']            # ordinal
passthrough_features  = []                    # nothing to pass through here

# Build the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  StandardScaler()),
        ]), numeric_features),

        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), low_card_cat_features),

        ('ord', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('enc',    OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), ordinal_features),
    ],
    remainder='drop',  # drop 'class' — redundant with pclass
    verbose_feature_names_out=True,
)

X_train_pre = preprocessor.fit_transform(X_train)
print(f'Output shape after ColumnTransformer: {X_train_pre.shape}')
print(f'Feature names:\n{preprocessor.get_feature_names_out()}')

---
## Section 4 — Full Preprocessing + Model Pipeline

In [ ]:
# Assemble the complete pipeline: preprocessing + classifier
full_pipe = Pipeline([
    ('pre',   preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42,
        n_jobs=-1,
    )),
])

# Fit on training data — all preprocessing is fit here, transform-only on test
full_pipe.fit(X_train, y_train)

# Evaluate on held-out test set
y_pred = full_pipe.predict(X_test)
print('Test set performance:')
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

In [ ]:
# Feature importances mapped back to meaningful names
feature_names = preprocessor.get_feature_names_out()
rf_model = full_pipe.named_steps['model']
importances = pd.Series(rf_model.feature_importances_, index=feature_names)

fig, ax = plt.subplots(figsize=(8, 5))
importances.sort_values().plot(kind='barh', ax=ax, color='#4a90d9', edgecolor='white')
ax.set_title('Feature importances (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## Section 5 — Cross-Validation with Pipeline (Zero Leakage)

In [ ]:
# cross_val_score with a Pipeline: for each fold, the full pipeline
# (preprocessor + model) is fit on the fold's train portion and evaluated on val
# → preprocessor never sees validation data during fit

cv_scores = cross_val_score(
    full_pipe, X_train, y_train,
    cv=5, scoring='accuracy', n_jobs=-1
)

print('5-fold cross-validated accuracy (no leakage):')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'  Mean : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Hyperparameter search across both preprocessing and model parameters
# Pipeline parameter names follow: stepname__paramname

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth'   : [5, 8, None],
}

grid_search = GridSearchCV(
    full_pipe, param_grid,
    cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print(f'Best params    : {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')
print(f'Test accuracy  : {grid_search.score(X_test, y_test):.4f}')

---
## Section 6 — Inspecting, Saving, and Loading a Pipeline

In [ ]:
# Inspect the pipeline structure
print('Pipeline steps:')
for name, step in full_pipe.steps:
    print(f'  [{name}] {type(step).__name__}')

print('\nPreprocessor transformers:')
for name, transformer, cols in preprocessor.transformers_:
    print(f'  [{name}] {type(transformer).__name__} → columns: {cols}')

# Access a specific step's learned parameters
scaler_in_pipe = full_pipe.named_steps['pre'].named_transformers_['num'].named_steps['scale']
print('\nStandardScaler means (first 3 features):', scaler_in_pipe.mean_[:3].round(4))

In [ ]:
# Save the fitted pipeline
model_path = Path('_saved_pipeline.joblib')
joblib.dump(full_pipe, model_path)
print(f'Pipeline saved to {model_path}  ({model_path.stat().st_size / 1e3:.1f} KB)')

# Load it back and verify predictions are identical
loaded_pipe = joblib.load(model_path)
y_pred_loaded = loaded_pipe.predict(X_test)
assert (y_pred == y_pred_loaded).all(), 'Predictions differ after reload!'
print('Loaded pipeline produces identical predictions: ✓')

# Clean up
model_path.unlink()

In [ ]:
# Predict on new, unseen data — everything applied automatically
new_passenger = pd.DataFrame([{
    'pclass': 2, 'sex': 'female', 'age': 28.0,
    'sibsp': 0, 'parch': 1, 'fare': 21.0,
    'embarked': 'S', 'class': 'Second'
}])

prediction = full_pipe.predict(new_passenger)
proba      = full_pipe.predict_proba(new_passenger)
print(f'Prediction      : {"Survived" if prediction[0] == 1 else "Died"}')
print(f'Survival probability: {proba[0][1]:.1%}')
print('\nAll preprocessing steps applied automatically — no manual transform needed.')

---
## Section 7 — Custom Transformers

Any domain-specific preprocessing step can be wrapped in a custom transformer by inheriting from `BaseEstimator` and `TransformerMixin`. This makes it a first-class citizen inside any Pipeline.

In [ ]:
class WinsorizeTransformer(BaseEstimator, TransformerMixin):
    """Cap feature values at specified percentile bounds."""

    def __init__(self, lower_pct: float = 0.01, upper_pct: float = 0.99):
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lower_bounds_ = X.quantile(self.lower_pct)
        self.upper_bounds_ = X.quantile(self.upper_pct)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for i, col in enumerate(X.columns):
            X[col] = X[col].clip(
                lower=self.lower_bounds_.iloc[i],
                upper=self.upper_bounds_.iloc[i],
            )
        return X.values


print('WinsorizeTransformer defined.')

In [ ]:
class FamilySizeTransformer(BaseEstimator, TransformerMixin):
    """Derive family_size = sibsp + parch + 1 and is_alone flag."""

    def fit(self, X, y=None):
        return self  # stateless

    def transform(self, X):
        X = pd.DataFrame(X, columns=['sibsp', 'parch'])
        family_size = X['sibsp'] + X['parch'] + 1
        is_alone    = (family_size == 1).astype(int)
        return np.column_stack([family_size.values, is_alone.values])


# Drop sibsp/parch from numeric features; let custom transformer produce family features
numeric_features_v2  = ['age', 'fare']
family_features      = ['sibsp', 'parch']

preprocessor_v2 = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('winsor', WinsorizeTransformer(lower_pct=0.01, upper_pct=0.99)),
            ('scale',  StandardScaler()),
        ]), numeric_features_v2),

        ('family', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value=0)),
            ('derive', FamilySizeTransformer()),
            ('scale',  StandardScaler()),
        ]), family_features),

        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), ['sex', 'embarked']),

        ('ord', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('enc',    OrdinalEncoder()),
        ]), ['pclass']),
    ],
    remainder='drop',
)

pipe_v2 = Pipeline([
    ('pre',   preprocessor_v2),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])

cv_v2 = cross_val_score(pipe_v2, X_train, y_train, cv=5, scoring='accuracy')
print(f'Pipeline v2 (with custom transformers) — CV accuracy: {cv_v2.mean():.4f} ± {cv_v2.std():.4f}')

---
## Putting It All Together — Regression Example

In [ ]:
# California Housing: a regression pipeline with transforms
housing = fetch_california_housing(as_frame=True)
df_h = housing.frame.copy()

X_h = df_h.drop(columns='MedHouseVal')
y_h = np.log1p(df_h['MedHouseVal'])  # log-transform target

X_h_tr, X_h_te, y_h_tr, y_h_te = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

# All features are numeric — single transformer branch
housing_pipe = Pipeline([
    ('pre', ColumnTransformer([
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('yj',     PowerTransformer(method='yeo-johnson')),
        ]), X_h.columns.tolist()),
    ])),
    ('model', GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05,
        max_depth=4, random_state=42
    )),
])

housing_pipe.fit(X_h_tr, y_h_tr)
r2 = r2_score(y_h_te, housing_pipe.predict(X_h_te))
print(f'California Housing — test R² (log price): {r2:.4f}')

---
## Key Takeaways

1. **Pipeline prevents the two most common preprocessing errors** — data leakage and training/serving skew — by construction.
2. **ColumnTransformer** lets you apply tailored transformations per column type within a single unified object.
3. **Cross-validation with Pipeline** is the only correct way to evaluate a model that includes preprocessing — each fold fits preprocessing only on its own train portion.
4. **GridSearchCV with Pipeline** allows hyperparameter search across both preprocessing and model parameters, denoted `stepname__param`.
5. **Custom transformers** (inheriting `BaseEstimator`, `TransformerMixin`) make any domain logic a reusable, pipeline-compatible step.
6. **`joblib.dump` / `joblib.load`** serialises the entire fitted pipeline — deploy it as one artefact; no manual state reconstruction.

---
## Exercises

1. Add a `LogTransformer` custom step that applies `log1p` to only the `fare` column before the `ColumnTransformer`, and show it improves cross-validated accuracy.
2. Extend the Titanic pipeline to also include the `deck` column (extracted from the cabin number). Handle its high missingness inside the Pipeline.
3. Use `GridSearchCV` to tune the imputation strategy (`median` vs `most_frequent`) for the age column alongside `RandomForest` hyperparameters. Use `Pipeline` parameter notation.
4. Demonstrate that fitting the scaler outside the Pipeline before `cross_val_score` inflates the CV score — show the numerical difference.

In [8]:
# ========== 完整 Pipeline 作业 (英文，稳健版) ==========
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')

# ========== 1. 加载数据 ==========
file_path = r'C:\Users\35111\Downloads\OneDrive_1_2026-5-26\titanic.csv'
titanic = pd.read_csv(file_path)
# 统一列名为小写
titanic.columns = [col.lower() for col in titanic.columns]
print("Columns found:", list(titanic.columns))

# 检查目标列
target = 'survived'
if target not in titanic.columns:
    raise KeyError(f"Target column '{target}' not found.")
X = titanic.drop(columns=[target])
y = titanic[target]

# 划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# ========== 2. 自定义转换器：对 fare 列应用 log1p ==========
class LogFareTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to the 'fare' column only."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        # X can be DataFrame or array; we convert to DataFrame to safely access column
        if isinstance(X, np.ndarray):
            # If input is array, we assume column order matches original
            # We need to know which column index is 'fare'
            # This requires storing column index during fit. Simpler: require X as DataFrame.
            # We'll convert back to DataFrame if needed.
            X = pd.DataFrame(X, columns=self.feature_names_in_)
        X = X.copy()
        if 'fare' in X.columns:
            X['fare'] = np.log1p(X['fare'])
        return X.values  # return as array for sklearn compatibility
    def get_feature_names_out(self, input_features=None):
        return input_features

# 注意：上面的转换器需要在 fit 时记住列名。更简单的方法是使用 FunctionTransformer 但针对数组操作。
# 我们采用更直接的方案：在 ColumnTransformer 外部预先处理 fare 列，或者使用一个安全的 lambda。
# 下面使用 FunctionTransformer 并指定列索引，但索引在不同数据集中可能变化。
# 最稳健：在 ColumnTransformer 的数值流水线中，对 fare 单独做 log1p。

# 定义特征列（基于小写列名）
feature_cols = [col for col in X.columns if col != target]
numeric_cols = [col for col in feature_cols if col in ['age', 'fare', 'sibsp', 'parch']]
categorical_cols = [col for col in feature_cols if col in ['sex', 'embarked']]
ordinal_cols = [col for col in feature_cols if col == 'pclass']
cabin_cols = [col for col in feature_cols if col == 'cabin']

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)
print("Ordinal:", ordinal_cols)
print("Cabin:", cabin_cols)

# 为了对 fare 单独做 log1p，我们可以在数值流水线中拆分 fare 和其他数值特征。
# 更好的办法：使用 ColumnTransformer 分别处理 fare 和其他数值列。
# 但为了简洁，我们创建一个新的列 'fare_log'，但这里直接做转换。
from sklearn.preprocessing import FunctionTransformer

def add_log_fare(X):
    X = X.copy()
    if 'fare' in X.columns:
        X['fare'] = np.log1p(X['fare'])
    return X

log_fare_transformer = FunctionTransformer(add_log_fare, validate=False)

# 构建预处理流水线
transformers = []

# 数值特征流水线（不包括 fare，因为 fare 已单独处理）
if numeric_cols:
    num_no_fare = [c for c in numeric_cols if c != 'fare']
    if num_no_fare:
        transformers.append(('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]), num_no_fare))

# fare 单独处理（log1p + scale）
if 'fare' in numeric_cols:
    transformers.append(('fare', Pipeline([
        ('log', FunctionTransformer(lambda X: np.log1p(X), validate=False)),
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), ['fare']))

# 分类特征
if categorical_cols:
    transformers.append(('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_cols))

# 序数特征 (pclass)
if ordinal_cols:
    transformers.append(('ord', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', sparse_output=False))
    ]), ordinal_cols))

# 船舱甲板特征
if cabin_cols:
    def extract_deck(X):
        X = X.copy()
        deck = X.iloc[:, 0].str[0].fillna('U')
        return pd.DataFrame(deck, columns=['deck'])
    transformers.append(('deck', Pipeline([
        ('extract', FunctionTransformer(extract_deck, validate=False)),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cabin_cols))

preprocessor = ColumnTransformer(transformers, remainder='drop')

# 最终 pipeline
pipeline = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(random_state=42, n_jobs=1))
])

# ========== Exercise 1: 交叉验证（使用 log1p on fare）==========
print("\nExercise 1: Cross-validation (cv=3) with log1p on fare")
scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='accuracy')
print(f"CV accuracy (with log1p): {scores.mean():.4f} ± {scores.std():.4f}")

# 对比：去掉 log1p（移除 fare 的 log 步骤）
# 构建一个没有 log 的 pipeline 做对比
transformers_no_log = []
if num_no_fare:
    transformers_no_log.append(('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), num_no_fare))
if 'fare' in numeric_cols:
    transformers_no_log.append(('fare', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), ['fare']))
if categorical_cols:
    transformers_no_log.append(('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_cols))
if ordinal_cols:
    transformers_no_log.append(('ord', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', sparse_output=False))
    ]), ordinal_cols))
if cabin_cols:
    transformers_no_log.append(('deck', Pipeline([
        ('extract', FunctionTransformer(extract_deck, validate=False)),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cabin_cols))

preprocessor_no_log = ColumnTransformer(transformers_no_log, remainder='drop')
pipeline_no_log = Pipeline([
    ('pre', preprocessor_no_log),
    ('clf', RandomForestClassifier(random_state=42, n_jobs=1))
])
scores_no_log = cross_val_score(pipeline_no_log, X_train, y_train, cv=3, scoring='accuracy')
print(f"CV accuracy (without log1p): {scores_no_log.mean():.4f} ± {scores_no_log.std():.4f}")

# ========== Exercise 2: 甲板提取已包含在上面的 pipeline 中 ==========
if cabin_cols:
    print("\nExercise 2: Deck extraction example (first 5 rows):")
    sample = X_train.head(5)[cabin_cols]
    deck_sample = extract_deck(sample)
    print(pd.concat([sample, deck_sample], axis=1).head())
else:
    print("\nExercise 2: Skipped because 'cabin' column not found.")

# ========== Exercise 3: GridSearchCV 调优年龄插补和随机森林参数 ==========
print("\nExercise 3: GridSearchCV (cv=3) for age imputation and RF params")
param_grid = {
    'pre__num__impute__strategy': ['median', 'most_frequent'],
    'clf__n_estimators': [50, 100],
    'clf__max_depth': [5, None]
}
# 注意：上面的 param_grid 中的 'num' 可能不包含 age？我们需要确保有 num 组件。
# 如果 num_no_fare 非空，则可以使用。否则跳过。
if num_no_fare:
    grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=1, verbose=0)
    grid_search.fit(X_train, y_train)
    print("Best params:", grid_search.best_params_)
    print("Best CV accuracy: {:.4f}".format(grid_search.best_score_))
    print("Test accuracy: {:.4f}".format(grid_search.score(X_test, y_test)))
else:
    print("No numeric features (except fare) found; skipping GridSearchCV.")

# ========== Exercise 4: 泄漏演示 ==========
print("\nExercise 4: Leakage demonstration")
# 泄漏做法：在整个 X_train 上 fit scaler，然后交叉验证（错误）
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[['age', 'fare', 'sibsp', 'parch']])
# 用缩放后的数据训练模型并交叉验证
model = RandomForestClassifier(random_state=42, n_jobs=1)
# 注意：这里我们没有将缩放集成到交叉验证中，而是预先缩放，这会导致数据泄漏。
# 为了模拟泄漏，我们直接使用缩放后的数据进行交叉验证。
scores_leaky = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='accuracy')
print(f"Leaky accuracy (scaler on full training set before CV): {scores_leaky.mean():.4f}")
# 正确做法（已在 pipeline 中）的结果
print(f"Correct accuracy (pipeline with internal scaling): {scores.mean():.4f}")

print("\nAll exercises completed.")

Columns found: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
Train: (712, 14), Test: (179, 14)
Numeric: ['age', 'sibsp', 'parch', 'fare']
Categorical: ['sex', 'embarked']
Ordinal: ['pclass']
Cabin: []

Exercise 1: Cross-validation (cv=3) with log1p on fare
CV accuracy (with log1p): 0.8006 ± 0.0151
CV accuracy (without log1p): 0.8006 ± 0.0151

Exercise 2: Skipped because 'cabin' column not found.

Exercise 3: GridSearchCV (cv=3) for age imputation and RF params
Best params: {'clf__max_depth': 5, 'clf__n_estimators': 50, 'pre__num__impute__strategy': 'median'}
Best CV accuracy: 0.8259
Test accuracy: 0.8212

Exercise 4: Leakage demonstration
Leaky accuracy (scaler on full training set before CV): 0.7191
Correct accuracy (pipeline with internal scaling): 0.8006

All exercises completed.
